### This script serves to test all 6 models on both datasets.
Note that the script needs to be run once per dataset (e.g. IISE 2024) to function. See the configuration options below.
It retrieves 10 results per model per standard user query. After this script is run, the resulting data is loaded into the LLM as a Judge script for evaluation.

#### Setup

In [18]:
import pandas as pd
import numpy as np
import datetime as dt
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from utils import split_into_sentences, merge_session_times
from utils import EmbModel
import time

In [19]:
#Configuration options
dataset = 'IISE 2024'
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
model_name_short = model_name_short = model_name.split('/')[-1]
dataset_name_nospace = dataset.replace(" ","")

In [20]:
session_filepath = f'./Datasets/{dataset}/session_times.csv'
embedding_filepath = f'./Datasets/{dataset}/Embeddings/{model_name_short}.pkl'
queries_filepath = f'./Datasets/{dataset}/Queries/testing_queries.xlsx'
presentation_filepath = f'./Datasets/{dataset}/{dataset_name_nospace}_Processed.csv'

In [21]:
embedding_model = SentenceTransformer(model_name)

emb = open(embedding_filepath, 'rb') 
embedding = pickle.load(emb)
emb.close()

if embedding.doChunk:
    pre_emb_path = f'./Datasets/{dataset}/chunked_df.csv'
else:
    pre_emb_path = f'./Datasets/{dataset}/non_chunked_df.csv'

presentation_df = pd.read_csv(presentation_filepath)
pre_embedding_df = pd.read_csv(pre_emb_path)
session_df = pd.read_csv(session_filepath)
queries_df = pd.read_excel(queries_filepath)

display(presentation_df.head(5))
display(pre_embedding_df.head(5))
display(session_df.head(5))
display(queries_df.head(5))

c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


,paper_title,abstract,session_name,session_id,paper_id
0,Optimizing Operations Managerial Performance: ...,Senior company leadership at an aircraft main...,Optimization,154,1
1,Discovering Puerto Rico Roads Profile Influenc...,The infrastructure of Puerto Rico (PR) was gi...,Logistics I,122,2
2,Designing Manufacturing Enterprises System for...,"Currently, researchers are concentrating thei...",Process Planning-I,180,3
3,Achieving Equitable Access to Medical Laborato...,"In contemporary healthcare, laboratory tests ...",Social Good Analytics,209,4
4,A Simulation-Based Case Study on ICU Patients ...,Intensive care units (ICUs) provide critical ...,Healthcare Optimization,87,5


,Unnamed: 0,paper_id,sentence,source
0,0,1,Optimizing Operations Managerial Performance: ...,Title
1,1,1,Senior company leadership at an aircraft main...,Abstract
2,2,1,Analysis of the data collected during the obse...,Abstract
3,3,1,"The findings provided valuable insights, highl...",Abstract
4,4,1,The brief study serves as the foundation for f...,Abstract


,Unnamed: 0,session_name,start_datetime,end_datetime,session_id,is_special_session,is_special_and_panel_session
0,0,AI in Health Systems,5/19/2024 12:00,5/19/2024 13:20,1,False,False
1,1,Advanced Data Analytics for Quality Control an...,5/19/2024 15:00,5/19/2024 16:20,2,False,False
2,2,Advanced Simulation Models,5/19/2024 16:30,5/19/2024 17:50,3,False,False
3,3,Advanced Topics in Smart Manufacturing I,5/19/2024 13:30,5/19/2024 14:50,4,False,False
4,4,Advanced Topics of QCRE Applications I,5/21/2024 8:00,5/21/2024 9:20,5,False,False


,Unnamed: 0,Prompt,Type
0,0,supply chain resilience strategies,AI gen - short
1,1,discrete event simulation applications,AI gen - short
2,2,ergonomics in manufacturing,AI gen - short
3,3,lean healthcare implementation,AI gen - short
4,4,data analytics for operations management,AI gen - short


#### Defining new testResult object

In [22]:
class testResult:
    def __init__(self, model_name_short, dataset, prompt, promptType):
        self.model_name_short = model_name_short
        self.dataset = dataset
        self.prompt = prompt
        self.promptType = promptType
        self.time = None
        self.result = None
        self.reranking_score_1 = None
        self.reranking_score_2 = None
        self.retrival_score = None

    def set_result_df(self, result):
        self.result = result

    def set_time(self, time):
        self.time = time  

    def set_reranking_1(self, reranking_score_1):
        self.reranking_score_1 = reranking_score_1

    def set_reranking_1(self, reranking_score_2):
        self.reranking_score_2 = reranking_score_2

    def set_retrieval(self, retrieval_score):
        self.retrieval_score = retrieval_score

#### Data Processing

In [23]:
#Rewriting get_best_abstracts function to take parameters
def get_best_abstracts(embeddings, lem_df, original_df, string, embedding_model):

    def get_best_embedding(string, embedding_model = embedding_model, embeddings = embeddings, lem_df = lem_df):
        str_embedding = embedding_model.encode(string)
        str_embedding_reshaped = str_embedding.reshape(1, -1)  # Reshape only once

        # Vectorized cosine similarity calculation
        cosine_similarities = cosine_similarity(np.array(embeddings), str_embedding_reshaped)
        cosine_similarities = cosine_similarities.flatten()  # Flatten to a 1D array

        lem_df['cosine_similarity'] = cosine_similarities

        test_df = lem_df.groupby(by='paper_id')['cosine_similarity'].mean()

        return test_df
    
#Splitting prompt into sentences
    prompt_df = pd.DataFrame()
    prompt_df['text'] = [string]
    if embeddings.doChunk:
        prompt_df = split_into_sentences(prompt_df)

    #Matching embeddings for each sentence
    agg_df = pd.DataFrame()

    #Splitting into sentences but if the model does not require chunking,
    #Then each 'sentence' will actually be the entire body of text
    for sentence in prompt_df['text']:
        sent_df = get_best_embedding(string=sentence, 
                                     embeddings = embeddings.embedding, 
                                     lem_df = lem_df,
                                     embedding_model=embedding_model)
        sent_df = pd.DataFrame(sent_df)
        sent_df.reset_index(names='paper_id', inplace=True)
        agg_df = pd.concat([agg_df, sent_df])

    #Aggregating
    agg_df = agg_df.groupby(by='paper_id').mean('cosine_similarity')

    #Here you can weigh the different cosine similarities somehow
    #Percentile mean code
    # else:
    #     agg_df = agg_df.groupby(by='cosine_similarity')
    #     agg_df = top_percentile_mean(agg_df, 'cosine_similarity', percentile= 0.6)
    #     agg_df.columns = ['cosine_similarity']

    output_df = agg_df.merge(original_df, left_index=True, right_on = 'paper_id')
    output_df.sort_values(by='cosine_similarity', ascending=False, inplace=True)

    output_df['text'] = output_df['paper_title'] + ". " + output_df['abstract']
    output_df = output_df[['text','cosine_similarity']]
    
    #TODO: output_df should contain cosine_similarity score and concatenated text comprised of paper_title and abstract
    return output_df

In [24]:
def get_result(embedding, pre_embedding_df, presentation_df, 
               model_name_short, emb_model, dataset, prompt_str, prompt_type):
    current_result = testResult(model_name_short=model_name_short,
                                dataset=dataset,
                                prompt=prompt_str,
                                promptType=prompt_type,
                                )

    start_time = time.perf_counter()

    result_df = get_best_abstracts(embeddings=embedding,
                            lem_df=pre_embedding_df,
                            original_df=presentation_df,
                            embedding_model=emb_model, 
                            string=current_result.prompt)

    current_result.set_result_df(result_df.head(10))

    end_time = time.perf_counter()
    total_time = end_time-start_time

    current_result.set_time(total_time)
    return current_result


#### Combined script

In [25]:
dataset_list = ['IISE 2024', 'INFORMS 2024']
model_list = ['Snowflake/snowflake-arctic-embed-m-v1.5',
                      'ibm-granite/granite-embedding-125m-english',
                      'Snowflake/snowflake-arctic-embed-s',
                      'BAAI/bge-small-en-v1.5',
                      'sentence-transformers/all-MiniLM-L12-v2',
                      'sentence-transformers/all-MiniLM-L6-v2']

result_list = []

for dataset in dataset_list:
    print('----------------------')
    print(f"Dataset: {dataset}")
    dataset_name_nospace = dataset.replace(" ","")
    session_filepath = f'./Datasets/{dataset}/session_times.csv'
    queries_filepath = f'./Datasets/{dataset}/Queries/testing_queries.xlsx'
    presentation_filepath = f'./Datasets/{dataset}/{dataset_name_nospace}_Processed.csv'

    presentation_df = pd.read_csv(presentation_filepath)
    session_df = pd.read_csv(session_filepath)
    queries_df = pd.read_excel(queries_filepath)

    for model in model_list:
        start_time_2 = time.perf_counter()
        print(f"Model: {model}")
        model_name_short = model.split('/')[-1]
        embedding_filepath = f'./Datasets/{dataset}/Embeddings/{model_name_short}.pkl'

        emb = open(embedding_filepath, 'rb') 
        embedding = pickle.load(emb)
        emb.close()
        embedding_model = SentenceTransformer(model)

        if embedding.doChunk:
            pre_emb_path = f'./Datasets/{dataset}/chunked_df.csv'
        else:
            pre_emb_path = f'./Datasets/{dataset}/non_chunked_df.csv'

        pre_embedding_df = pd.read_csv(pre_emb_path)

        for index, row in queries_df.iterrows():
            prompt_str = row['Prompt']
            prompt_type = row['Type']
            result = get_result(embedding=embedding,
                                pre_embedding_df=pre_embedding_df,
                                presentation_df=presentation_df,
                                emb_model = embedding_model,
                                model_name_short=model_name_short,
                                dataset=dataset,
                                prompt_str=prompt_str,
                                prompt_type=prompt_type)
            result_list.append(result)
        end_time_2 = time.perf_counter()
        print(f"Testing time: {end_time_2 - start_time_2}")
        print(' ')


----------------------
Dataset: IISE 2024
Model: Snowflake/snowflake-arctic-embed-m-v1.5
Testing time: 7.430001199943945
 
Model: ibm-granite/granite-embedding-125m-english
Testing time: 8.653189899981953
 
Model: Snowflake/snowflake-arctic-embed-s
Testing time: 5.034050400019623
 
Model: BAAI/bge-small-en-v1.5
Testing time: 6.507382799987681
 
Model: sentence-transformers/all-MiniLM-L12-v2


c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Testing time: 12.935570800094865
 
Model: sentence-transformers/all-MiniLM-L6-v2


c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Testing time: 9.22792540001683
 
----------------------
Dataset: INFORMS 2024
Model: Snowflake/snowflake-arctic-embed-m-v1.5
Testing time: 10.645697900094092
 
Model: ibm-granite/granite-embedding-125m-english
Testing time: 12.826065300032496
 
Model: Snowflake/snowflake-arctic-embed-s
Testing time: 6.23559109994676
 
Model: BAAI/bge-small-en-v1.5
Testing time: 6.342590200016275
 
Model: sentence-transformers/all-MiniLM-L12-v2


c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Testing time: 23.340994599973783
 
Model: sentence-transformers/all-MiniLM-L6-v2


c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Testing time: 18.499452500022016
 


In [27]:
filename = r'.\testing_files\test_results'

with open (filename, 'wb') as f:
      pickle.dump(result_list, f)